In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.metrics import mean_squared_error
color_pal = sns.color_palette()
plt.style.use('fivethirtyeight')

In [ ]:
df = pd.read_csv('/kaggle/input/delhi-hourly-2023/final_modified.csv')
df = df.set_index('Date and Time')
df.index = pd.to_datetime(df.index)

In [ ]:
df.plot(style='.',
        figsize=(15, 10),
        color=color_pal,
        title='2023 hourly demand',
       legend = False)
plt.show()

In [ ]:


df = pd.read_csv('/kaggle/input/delhi-hourly-2023/final_modified.csv')
df = df.set_index('Date and Time')
df.index = pd.to_datetime(df.index)
# Scale down the first column by dividing by 1000
first_column_name = df.columns[2]  # Automatically get the name of the first column
df[first_column_name] = df[first_column_name] / 100 #scaled down by 100

temp = df.columns[3] #scaled Temp by 1000
df[temp] = df[temp] * 100

humid = df.columns[6]
df[humid] = df[humid] * 20

solRad = df.columns[16]
df[solRad] = df[solRad] * 3

solEn = df.columns[17]
df[solEn] = df[solEn] *100

# Define colors for each column
# Replace 'color_pal' with your custom color palette
defcolor = (0,0,0,0)

color_pal = [defcolor] * len(df.columns)  # Set all to gray initially
color_pal[0] = 'red'  # Highlight the first column in red
color_pal[1] = 'blue'  # Highlight the Temp column in blue
color_pal[4] = 'green'  # Highlight the Humid column in green
color_pal[8] = 'orange'  # Highlight the SolRad column in orange
color_pal[13] = 'black'  # Highlight the SolRad column in orange


# Plot the data
df.plot(style='.',
        figsize=(15, 10),
        color=color_pal,
        title='2023 hourly demand')



# Display the plot
plt.show()



In [ ]:
color_pal[5] = 'green'  # Highlight the SolRad column in orange
color_pal[9] = 'orange'  # Highlight the SolRad column in orange

color_pal[14] = 'black'  # Highlight the SolRad column in orange
#color_pal[15] = 'brown'




df.loc[(df.index > '01-01-2023') & (df.index < '01-31-2023')] \
    .plot(figsize=(70, 20),style = "-",color=color_pal, title='Week Of Data',
         legend = False)

plt.show()

In [ ]:
train = df.loc[(df.index < '2023-12-31') & (df.index >= '2023-04-01')]  # Corrected date format and logic
test = df.loc[df.index < '2023-03-01']

fig, ax = plt.subplots(figsize=(15, 5))
train.plot(ax=ax, label='Training Set', title='Data Train/Test Split')
test.plot(ax=ax, label='Test Set')
ax.axvline('01-01-2015', color='black', ls='--')
ax.legend(['Training Set', 'Test Set'])
plt.show()

In [ ]:

def extract_weather_features(df):
    """
    Extract specific weather-related features from the DataFrame.
    
    Parameters:
    df (pandas.DataFrame): The input DataFrame with weather data.
    
    Returns:
    pandas.DataFrame: A DataFrame with the selected features.
    """
    # List of specific weather-related features to extract
    features = ['temp', 'humidity', 'solarradiation', 'windspeed']
    
    # Check if all features are in the DataFrame
    available_features = [feature for feature in features if feature in df.columns]
    
    # Extract and return the available features
    extracted_df = df[available_features].copy()
    
    return extracted_df

# Apply the function to your DataFrame
extracted_features_df = extract_weather_features(train)

# Display the first few rows of the extracted features
extracted_features_df.head()


In [ ]:
# Extract features from the train and test datasets
X_train = extract_weather_features(train)
X_test = extract_weather_features(test)

# Assuming 'temp' is the target variable for regression
y_train = train['temp']
y_test = test['temp']

# Initialize the XGBoost regressor
regressor = xgb.XGBRegressor(
    n_estimators=100000,
    learning_rate=0.01,
    max_depth=3,
    objective='reg:squarederror',
    early_stopping_rounds=50,
    eval_metric='rmse',
    random_state=42
)

# Train the model
regressor.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

# Make predictions
y_pred = regressor.predict(X_test)

# Evaluate the model
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'Root Mean Squared Error: {rmse}')

# Optional: Feature importance plot
xgb.plot_importance(regressor)
plt.show()

In [ ]:
y_pred = regressor.predict(X_test)

# Evaluate the model
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'Root Mean Squared Error: {rmse}')

# Plot the predictions vs the actual values
plt.figure(figsize=(10, 6))
plt.plot(y_test.index, y_test, label='Actual', color='blue')
plt.plot(y_test.index, y_pred, label='Predicted', color='red', linestyle='--')
plt.xlabel('Date and Time')
plt.ylabel('Temperature')
plt.title('Predicted vs Actual Temperatures')
plt.legend()
plt.show()